In the *Common Corpus* [preprint](https://arxiv.org/pdf/2506.01732) (CC, @langlais_2025), the authors write (p. 5n2) that the token counts they report are based on the "Pleias base tokenizer", that is the tokenizer used with the "small language model" [Pleias-pico-350m-Preview](https://huggingface.co/PleIAs/Pleias-350m-Preview) (referred to below as Pleias 1.0).

This tokenizer is a [byte-pair encoding](https://en.wikipedia.org/wiki/Byte-pair_encoding) (BPE) tokenizer and is available as a JSON file in the model repository. BPE is a subword tokenization strategy—we create an efficient vocabulary from frequently appearing parts of words in the training data. Words can then be reconstituted from their subwords. Accordingly, the number of tokens in a text will outnumber the number of words.

We will see this in some more detail below, but quickly here are three Latin words and how they are represented after tokenization with Pleias 1.0:

- est → 'Ġest'
- omnis → 'Ġomnis'
- divisa →  'Ġdiv', 'isa'

Frequently occurring words (like *est* and *omnis*) are represented as single tokens, while less frequent words (like *divisa*) are split into subwords. The prefixed 'Ġ' indicates that the token is the start of a new word.

Our goal in this post is to use this tokenizer on a handful of Latin texts of varying type and quality to get a sense of the ratio of words to Pleias tokens. This will allow us to estimate better the number of words corresponding to the 36 billion tokens advertised in the preprint. 

<!-- <span class='.preview-image' style="display: none;"><img src='preview.png'></img></span> -->

<!-- **Run this notebook in the browser using Binder here [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/diyclassics/epblog/notebooks?labpath=notebooks%2Fpleias-tokenizer.ipynb)** -->

## Code

First we need to download the tokenizer JSON file from the Pleias 1.0 repository...

In [1]:
# Download tokenizer.json from Pleias 1.0 (Pleias-350m-Preview)

import os
import requests

# Create the data directory if it doesn't exist
os.makedirs("data", exist_ok=True)

url = "https://huggingface.co/PleIAs/Pleias-350m-Preview/raw/main/tokenizer.json"
dest_path = os.path.join("data", "tokenizer.json")

# Download the file
response = requests.get(url)
with open(dest_path, "wb") as f:
    f.write(response.content)

print(f"Downloaded tokenizer.json to {dest_path}")

Downloaded tokenizer.json to data/tokenizer.json


Let's know show the output of this tokenizer on a sample Latin sentence, both the encoding and decoding...

In [2]:
# Tokenize example sentence with Pleias 1.0 tokenizer

from transformers import PreTrainedTokenizerFast

text = "Gallia est omnis divisa in partes tres."

Tok = PreTrainedTokenizerFast(tokenizer_file="data/tokenizer.json")

tokens = Tok.tokenize(text)
ids = Tok.convert_tokens_to_ids(tokens)
decoded_texts = Tok.decode(ids)

print("\nExample sentence:", text)
print("Token strings:", tokens)
print("Token ids:", ids)
print("Decoded text:", decoded_texts)


Example sentence: Gallia est omnis divisa in partes tres.
Token strings: ['Gall', 'ia', 'Ġest', 'Ġomnis', 'Ġdiv', 'isa', 'Ġin', 'Ġpartes', 'Ġtres', '.']
Token ids: [50057, 461, 706, 61508, 3055, 8767, 279, 17710, 13135, 17]
Decoded text: Gallia est omnis divisa in partes tres.


Now let's expand this to a longer, high-quality Latin text from the CLTK-Tesserae corpus (Cic. Cat. 1). Note that the CLTK `words` method uses the LatinCy tokenizer and so the output includes puncuation and similar non-word tokens. For our purposes in this post, what is important is that the LatinCy tokenizer is not a subword tokenizer like the Pleias tokenizer.

In [3]:
# Load high-quality text, i.e. Cicero's Cat. 1 from CLTK-Tesserae

# Temp fix for running multiple tokenizers
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from cltkreaders.lat import LatinTesseraeCorpusReader

T = LatinTesseraeCorpusReader()

file = 'cicero.in_catilinam.tess'
cicero = T.fileids(match=file)
text = next(T.texts(cicero))
print("Opening of Cat. 1:")
print(text[:100])  # first 100 characters
words = list(T.words(cicero))
print(f"Number of words in Cat. 1: {len(words)}")

Opening of Cat. 1:
quo usque tandem abutere, Catilina, patientia nostra? quam diu etiam furor iste tuus nos eludet? que
Number of words in Cat. 1: 14714


Now let's compare the number of words given by the LatinCy tokenizer to the number of Pleias tokens for the same text...

In [4]:
# Tokenize Cicero's Cat. 1 with Pleias tokenizer and compute ratio of tokens to words

tokens = Tok.tokenize(text)

print("Word-Token summary for Cat. 1:")
print(f"Number of words: {len(words)}")
print(f"Number of tokens: {len(tokens)}")
ratio = len(tokens) / len(words)
print(f"Tokens/words ratio: {ratio:.2f}")

Word-Token summary for Cat. 1:
Number of words: 14714
Number of tokens: 23778
Tokens/words ratio: 1.62


If the CLTK-Tesserae text were representative of the Latin texts in CC, and we used this tokens-to-words ratio (1.62) to estimate the CC word count...

In [5]:
# Calculate the number of words corresponding to 36 billion tokens using the observed ratio

pleias_latin_tokens = 36_000_000_000
estimated_words = pleias_latin_tokens / ratio
print(f"Estimated number of Latin words for 36B Latin Pleias tokens: {estimated_words:,.0f}")

Estimated number of Latin words for 36B Latin Pleias tokens: 22,277,062,831


But most of the content in CC is lower quality than our CLTK-Tesserae version of Cicero. Here are some extracts from an Internet Archive text (Ratellerus's 1576 edition of *Trageodiae Sophoclis*):

In [6]:
# Load lower-quality text from Internet Archive...

from bs4 import BeautifulSoup

noisy_url = "https://archive.org/stream/bub_gb_u1HTSd5V6dwC/bub_gb_u1HTSd5V6dwC_djvu.txt"

response = requests.get(noisy_url)
noisy_text = response.text

soup = BeautifulSoup(noisy_text, "html.parser")
main = soup.find("pre")
if main:
    main_plaintext = main.get_text(separator="\n").strip()
    with open("data/noisy_latin.txt", "w", encoding="utf-8") as out:
        out.write(main_plaintext)
else:
    print("No <main id=\"maincontent\"> block found.")

print("Extract 1 from noisy text:")
print(main_plaintext[0:250])

Extract 1 from noisy text:
z/^  -i,  ■ 


2^ 


iXsl^ 


"II 


:•>:  5-  ^ 


; • 


:»V 


_ ^ , _ ._  . . 

|it'-’ 

..  ^ ■ -♦<  jf>  ^iL-  ■ 


; s 


I-' ^ >»  ■i';'  , 7^-.'^  ;-'■>• 

' ' ..  - * *,  - ‘T  ■' 


r^ 


¥3kj 


•■■K- ' V 


, . . . 9-  * ■ ’ 


V ■« 


/


In [7]:
print("Extract 2 from noisy text:")
print(main_plaintext[20000:20250])

Extract 2 from noisy text:
rans  ambitione  ruit  ? 

Tcflis  Flyffes  erit  cum  Principe  claflis  Achiua, 
Donec  UHaonidls  nobile  viuet  opus. 

Nam  fefe procet  es  credens  iugulare  Pelajgos,  , 

Dicitur  armentis  commaculajfe  manus. 

Hinc  ira  impatiens  Grace 


In [8]:
print("Extract 3 from noisy text:")
print(main_plaintext[50000:50250])

Extract 3 from noisy text:
itur, 
Sapeaparuismagnorumres  ,,  .i* 
Confirmantur.  ^ , ^ . 

Sedvacors  hac  noncapit  animus. 

Turbas  id  genus  hondnes, 

exagitaris,  nunc  concitant. 

Neque  nos,  AiaXytefine  , 

Hancvim  tropulfarevalemus, 

AfieUurnnifimulacfmm 

V i'


We'll use `split` to tokenize this text approximately into words, and again compare this to the Pleias tokenizer output...

In [9]:
# Tokenize Ratellerus with Pleias tokenizer and compute ratio of tokens to words

words = main_plaintext.split()
tokens = Tok.tokenize(main_plaintext)

print("Word-Token summary for Ratellerus extract:")
print(f"Number of words: {len(words)}")
print(f"Number of tokens: {len(tokens)}")
ratio = len(tokens) / len(words)
print(f"Tokens/words ratio: {ratio:.2f}")

Word-Token summary for Ratellerus extract:
Number of words: 101096
Number of tokens: 330870
Tokens/words ratio: 3.27


As we can see, the noisy text has a much higher token-to-word ratio, almost certainly because of the large amount of noise (i.e. non-word OCR artifacts) in the digitized text.

Before coming to any provisional conclusions, let's try one more text, one of reasonably good quality but also with non-standard Latin words and one drawn from material known to be in CC. Here is the first text alphabetically in the [Latin Wikipedia, or *Vicipaedia*](https://la.wikipedia.org/), namely the article on 'Weird Al' Yankovic.

In [10]:
import requests
from bs4 import BeautifulSoup

vicipedia_url = "https://la.wikipedia.org/wiki/Alfredus_Yankovic"

response = requests.get(vicipedia_url)
soup = BeautifulSoup(response.text, "html.parser")

content_div = soup.find("div", {"id": "mw-content-text"})
if content_div:
    article_text = content_div.get_text(separator=" ", strip=True)
else:
    article_text = ""

print(article_text)

-4 (corrigenda) Latinitas huius paginae corrigenda est. Si potes, corrige vel rescribe. Vide {{ latinitas }}. Alfredus Yankovic (usitate notus a suo agnomine "Weird Al" Yankovic ( Latine "Alienus Al")) est cantor Americanus carminum satiricorum . Notus est per paroediis suis carminium popularium Michaelis Jackson , Madonnae , et aliorum cantorum famosorum. Vita [ recensere | fontem recensere ] Natus 23 Octobris 1959 in Lynwood in California . [ 1 ] Quando 7 anni natus est, mercator vendere eo accordion aut citharam obtulit. Pater Alfredi accordion decrevit quod fidicinem musicae polcae alium cum agnomine Yankovic voluit (primus erat "rex polcae" Franciscus Yankovic). [ 2 ] Alfredus non cognatus est eo, sed erat unus suorum inspirationum). Erat valedictor in suo lyceo, ab ubi graduavit uno anno mature. Suam careram musicam incepit quando erat discipulus in Universitate Polytechnica Californiae in Sancto Ludovico Episcopo. [ 3 ] Erat in hoc tempore quando apparuit on programmate radiopho

How does this Latin (if not canonical Latin) text fare in comparison to the Cicero and Ratellerus texts?

In [11]:
# Tokenize Vicipaedia article with Pleias tokenizer and compute ratio of tokens to words

words = article_text.split()
tokens = Tok.tokenize(article_text)

print("Word-Token summary for Vicipaedia extract:")
print(f"Number of words: {len(words)}")
print(f"Number of tokens: {len(tokens)}")
ratio = len(tokens) / len(words)
print(f"Tokens/words ratio: {ratio:.2f}")

Word-Token summary for Vicipaedia extract:
Number of words: 238
Number of tokens: 484
Tokens/words ratio: 2.03


We can now summarize our findings...

In [12]:
# Summarize findings

from tabulate import tabulate

data = [
    ["Cicero", 14714, 23778, 1.62],
    ["Vicipaedia", 238, 484, 2.03],
    ["Ratellerus", 101096, 330870, 3.27],
]

headers = ["Text", "Words", "Tokens", "Tokens/Words Ratio"]

print(tabulate(data, headers=headers))

Text          Words    Tokens    Tokens/Words Ratio
----------  -------  --------  --------------------
Cicero        14714     23778                  1.62
Vicipaedia      238       484                  2.03
Ratellerus   101096    330870                  3.27


Let's provisionally establish an lower and upper bound for the token-to-word ratio in Latin texts. In the case of the higher-quality text, we have 1.62 tokens per word. In the case of the lower-quality text, we have 3.27 tokens per word. Since CC reports 36 billion tokens, this suggests a range of 11 to 22.2 billion words in CC Latin texts, and we should err for now on the lower-bound side of this range as assessment of the CC Latin text quality continues.

In [13]:
pleias_latin_tokens = 36_000_000_000
lower_ratio = 1.62  # from Cicero
upper_ratio = 3.27  # from noisy text

lower_estimate = pleias_latin_tokens / upper_ratio
upper_estimate = pleias_latin_tokens / lower_ratio

print(f"Estimated Latin words in CC (provisional): {lower_estimate/1_000_000_000:.1f}B to {upper_estimate/1_000_000_000:.1f}B")

Estimated Latin words in CC (provisional): 11.0B to 22.2B


## References

::: {#refs}
:::